# Module 4: Applying Standards in Practice — Harmonizing Multi-Source Regional Data

**Unit B · Week 4** · Track 1 — Data Integration, Standards, Metadata & Quality

The first module where all three of this track's raw sources meet: joined on P-codes rather than names, dates normalized to ISO 8601, and a HXL tag row added to the published intermediate file.

## Learning objectives

- Add HXL hashtags to a raw dataset.
- Join disparate regional datasets using P-codes instead of place names.
- Normalize dates and country/district codes to ISO standards, and validate join coverage rather than silently dropping unmatched rows.


## Setup

This notebook reads the raw practice files in `../../../data/raw/`, built by
`data/make_track1_sources.py` — three partner-style exports shaped like a
real regional hub's source-system landscape (an ERP/financial export, a
survey-platform export, and an HDX-style pull), plus the P-code gazetteer
used to reconcile them. All four are **synthetic**; see `data/README.md`.
Run `python3 data/make_track1_sources.py` once from the repo root before
working through this notebook if those files aren't there yet.

## Lesson content

See the Python notebook for the full lesson content — identical in both languages.

In [ ]:
library(tidyverse)

gaz <- read_csv("../../../data/raw/cod_ab_gazetteer.csv", show_col_types = FALSE)
a <- read_csv("../../../data/raw/partner_a_finance_export.csv", show_col_types = FALSE)
b <- read_csv("../../../data/raw/partner_b_survey_export.csv", show_col_types = FALSE)

unmatched_a_raw <- a %>% filter(!reg %in% gaz$district_name)
unmatched_b_raw <- b %>% filter(!region_name %in% gaz$district_name)
cat("Partner A:", nrow(unmatched_a_raw), "row(s) unmatched before correction\n")
cat("Partner B:", nrow(unmatched_b_raw), "row(s) unmatched before correction\n")

In [ ]:
ALIASES <- c("Nyarugenge Dist." = "Nyarugenge", "Rwamagana " = "Rwamagana")

a <- a %>% mutate(reg = recode(str_trim(reg), !!!ALIASES))
b <- b %>% mutate(region_name = recode(str_trim(region_name), !!!ALIASES))

still_unmatched <- bind_rows(
  a %>% filter(!reg %in% gaz$district_name),
  b %>% filter(!region_name %in% gaz$district_name)
)
cat("Still unmatched after alias correction:", nrow(still_unmatched), "row(s)\n")

In [ ]:
a <- a %>%
  mutate(date = as.Date(date, format = "%d/%m/%Y")) %>%
  left_join(gaz, by = c("reg" = "district_name"))

b <- b %>%
  mutate(date = as.Date(collection_date, format = "%d-%b-%y")) %>%
  left_join(gaz, by = c("region_name" = "district_name"))

head(a %>% select(district_pcode, date, rev))

In [ ]:
harmonized <- a %>%
  transmute(district = reg, district_pcode, date = format(date, "%Y-%m-%d"), value = rev)

hxl_row <- tibble(district = "#adm2+name", district_pcode = "#adm2+code",
                   date = "#date", value = "#value+funding")
hxl_tagged <- bind_rows(hxl_row, harmonized)
write_csv(hxl_tagged, "../../../data/processed/harmonized_regional_data.csv")
cat("Wrote", nrow(harmonized), "rows (plus 1 HXL tag row) to harmonized_regional_data.csv\n")

## Your turn

Confirm both misspelled rows (one in Partner A, one in Partner B) are caught by the before/after unmatched-row check, then extend the harmonization to also bring in Partner C's `indicator` and `outcome` columns, joined on `district_pcode` and `date`.

**Formative assessment.** Submitted harmonized, HXL-tagged CSV, graded on join correctness (unmatched rows flagged, not dropped), correct hashtags, and ISO-format dates.